# Descriptive analysis

## 2. Övergripande skadebild

Det här avsnittet sammanfattar hur ovanliga skador är i träningsdatan och sätter skadefrekvensen i relation till både antal rader och exponeringsår.

In [1]:
from pathlib import Path

import numpy as np

import pandas as pd

data_path = Path("../data/Entreprenadförsäkring training.csv")
df = pd.read_csv(data_path)

claim_share = (
    df["AntalSkador"]
    .value_counts(normalize=True)
    .sort_index()
    .rename("andel")
    .mul(100)
)

total_claims = int(df["AntalSkador"].sum())
total_duration = df["Duration"].sum()
claims_per_row = df["AntalSkador"].mean()
claims_per_exposure_year = total_claims / total_duration

claim_summary = pd.DataFrame(
    {
        "Mått": [
            "Andel med 0 skador (%)",
            "Andel med 1 skada (%)",
            "Andel med 2 skador (%)",
            "Totalt antal skador",
            "Totalt antal exponeringsår",
            "Skador per rad",
            "Skador per exponerat år",
        ],
        "Värde": [
            round(claim_share.get(0, 0), 2),
            round(claim_share.get(1, 0), 2),
            round(claim_share.get(2, 0), 2),
            total_claims,
            round(total_duration, 0),
            round(claims_per_row, 4),
            round(claims_per_exposure_year, 5),
        ],
    }
)

claim_summary


,Mått,Värde
0,Andel med 0 skador (%),98.12000
1,Andel med 1 skada (%),1.86000
2,Andel med 2 skador (%),0.03000
3,Totalt antal skador,19730.00000
4,Totalt antal exponeringsår,924039.00000
5,Skador per rad,0.01910
6,Skador per exponerat år,0.02135


### Tolkning

Skador är ovanliga i datan:

- 98,12 % av raderna har 0 skador
- 1,86 % har 1 skada
- 0,03 % har 2 skador

Totalt i träningsdatan finns:

- 19 730 skador
- 924 039 exponeringsår summerat via `Duration`
- 0,0191 skador per rad
- 0,02135 skador per exponerat år

Det betyder att skadefrekvensen är låg, vilket är viktigt för hela projektet. Det antyder att vi arbetar med count-data med många nollor, vilket passar bra med enklare frekvensmodeller som Poisson eller eventuellt negativ binomial senare.

## 3. Beskrivning av de numeriska variablerna

Här sammanfattas de viktigaste numeriska variablerna med fokus på typiska nivåer och hur extrema observationerna är.

In [ ]:
numeric_summary = pd.DataFrame(
    {
        "Variabel": [
            "Omsättning",
            "Försäkringsbelopp",
            "Självrisk",
            "Duration",
        ],
        "Medel": [
            round(df["Omsattning"].mean(), 0),
            round(df["Forsakringsbelopp"].mean(), 0),
            round(df["Sjalvrisk"].mean(), 0),
            round(df["Duration"].mean(), 3),
        ],
        "Median": [
            round(df["Omsattning"].median(), 0),
            round(df["Forsakringsbelopp"].median(), 0),
            round(df["Sjalvrisk"].median(), 0),
            round(df["Duration"].median(), 1),
        ],
        "99-percentil": [
            round(df["Omsattning"].quantile(0.99), 0),
            round(df["Forsakringsbelopp"].quantile(0.99), 0),
            round(df["Sjalvrisk"].quantile(0.99), 0),
            round(df["Duration"].quantile(0.99), 1),
        ],
        "Max": [
            round(df["Omsattning"].max(), 0),
            round(df["Forsakringsbelopp"].max(), 0),
            round(df["Sjalvrisk"].max(), 0),
            round(df["Duration"].max(), 1),
        ],
    }
)

duration_full_year_share = (df["Duration"] == 1).mean() * 100

numeric_summary


### Tolkning

**Omsättning**

- Medel: 13,38 miljoner SEK
- Median: 8,10 miljoner SEK
- 99-percentil: 83,47 miljoner SEK
- Max: 1,22 miljarder SEK

**Försäkringsbelopp**

- Medel: 1,06 miljoner SEK
- Median: 507 000 SEK
- 99-percentil: 8,55 miljoner SEK
- Max: 119,9 miljoner SEK

**Självrisk**

- Medel: 16 106 SEK
- Median: 10 000 SEK
- 99-percentil: 50 000 SEK
- Max: 250 000 SEK

**Duration**

- Medel: 0,894
- Median: 1,0
- Cirka 73 % av observationerna gäller hela året

De ekonomiska variablerna är tydligt högerskevda, med ett litet antal mycket stora företag eller försäkringar. Det talar för att vi senare bör överväga log-transformering av exempelvis omsättning och försäkringsbelopp i modelleringen.

## 4. Fördelning mellan verksamheter

I det här avsnittet jämförs både portföljens sammansättning och skadefrekvensen mellan olika verksamhetstyper.

In [ ]:
business_share = (
    df["Verksamhet"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("Verksamhet")
    .reset_index(name="Andel av portföljen (%)")
)

business_claim_rate = (
    df.groupby("Verksamhet", observed=False)
    .agg(
        Skador=("AntalSkador", "sum"),
        Exponeringsar=("Duration", "sum"),
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Skador"] / x["Exponeringsar"]})
    .sort_values("Skador per exponerat år", ascending=False)
    .reset_index()
)

business_share


### Tolkning

Största verksamhetsgrupperna i träningsdatan är:

- Byggföretag: 39,96 %
- Övriga specialistföretag: 16,99 %
- Målare: 10,05 %
- Elektriker: 9,99 %
- VVS: 9,98 %

När man jämför skadefrekvens per exponerat år får man tydliga skillnader:

| Verksamhet | Skador per exponerat år |
| --- | ---: |
| VVS | 0,0316 |
| Takarbeten | 0,0250 |
| Byggföretag | 0,0221 |
| Övriga specialistföretag | 0,0214 |
| Grävning & Schaktning | 0,0189 |
| Elektriker | 0,0154 |
| Målare | 0,0141 |

Det finns alltså tydliga skillnader i risk mellan verksamhetstyper. VVS sticker ut mest, följt av takarbeten. Målare och elektriker har lägst skadefrekvens. Det här är ett starkt argument för att verksamhet ska vara en central variabel i både den deskriptiva analysen och den senare modellen.

## 5. Fördelning mellan geografiska områden

Här analyseras hur portföljen är fördelad mellan geografiska områden och hur skadefrekvensen varierar mellan dessa grupper.

In [ ]:
geography_share = (
    df["GeografisktOmrade"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("Geografiskt område")
    .reset_index(name="Andel av portföljen (%)")
)

geography_claim_rate = (
    df.groupby("GeografisktOmrade", observed=False)
    .agg(
        Skador=("AntalSkador", "sum"),
        Exponeringsar=("Duration", "sum"),
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Skador"] / x["Exponeringsar"]})
    .sort_values("Skador per exponerat år", ascending=False)
    .reset_index()
    .rename(columns={"GeografisktOmrade": "Geografiskt område"})
)

geography_share


### Tolkning

Andel av portföljen:

- Storstad: 40,0 %
- Mellanstorstad: 30,0 %
- Småstad: 20,0 %
- Landsbyggd: 10,0 %

Skadefrekvens per exponerat år:

| Geografiskt område | Skador per exponerat år |
| --- | ---: |
| Storstad | 0,0261 |
| Mellanstorstad | 0,0215 |
| Landsbyggd | 0,0179 |
| Småstad | 0,0135 |

Storstad har klart högst skadefrekvens, medan småstad ligger lägst. Det tyder på att marknadstyp eller arbetsmiljö skiljer sig åt på ett sätt som påverkar risknivån. Geografi verkar alltså också vara en viktig förklaringsfaktor.

## 6. Skillnader över tid

Här studeras hur skadefrekvensen utvecklas mellan kalenderåren i träningsdatan.

In [ ]:
year_summary = (
    df.groupby("Ar", observed=False)
    .agg(
        **{
            "Antal rader": ("Ar", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
    .rename(columns={"Ar": "År"})
)

year_summary["År"] = year_summary["År"].astype(int)
year_summary[["År", "Antal rader", "Totala skador", "Skador per exponerat år"]]


### Tolkning

Skadefrekvens per exponerat år per kalenderår:

| År | Antal rader | Totala skador | Skador per exponerat år |
| --- | ---: | ---: | ---: |
| 2021 | 239 213 | 4 565 | 0,02134 |
| 2022 | 252 034 | 4 627 | 0,02053 |
| 2023 | 264 444 | 5 092 | 0,02154 |
| 2024 | 277 695 | 5 446 | 0,02193 |

Det finns ingen dramatisk trend, men en svag rörelse uppåt från 2022 till 2024. Samtidigt växer portföljen i antal observationer och exponering. Detta stödjer uppgiftsbeskrivningens poäng om att skadebeteenden kan förändras över tid och att år därför bör finnas med i modellen.

## 7. Duration och exponering

I det här avsnittet undersöks om skadefrekvensen är stabil när observationerna delas upp efter hur stor del av året försäkringen varit aktiv.

In [ ]:
duration_bins = [0, 0.25, 0.5, 0.75, 0.99, 1.0]
duration_labels = [
    "0 till 0,25 år",
    "0,25 till 0,5 år",
    "0,5 till 0,75 år",
    "0,75 till 0,99 år",
    "0,99 till 1,0 år",
]

duration_band = pd.cut(
    df["Duration"],
    bins=duration_bins,
    labels=duration_labels,
    include_lowest=True,
    right=True,
)

duration_summary = (
    df.assign(DurationBand=duration_band)
    .groupby("DurationBand", observed=False)
    .agg(
        **{
            "Antal rader": ("Duration", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
    .rename(columns={"DurationBand": "Duration-intervall"})
)

duration_summary[["Duration-intervall", "Antal rader", "Totala skador", "Skador per exponerat år"]]


### Tolkning

När man delar upp efter hur stor del av året försäkringen varit aktiv blir skadefrekvensen per exponerat år ganska stabil:

- 0 till 0,25 år: 0,0192
- 0,25 till 0,5 år: 0,0219
- 0,5 till 0,75 år: 0,0207
- 0,75 till 0,99 år: 0,0216
- 0,99 till 1,0 år: 0,0214

Det här är faktiskt ett bra tecken. Det tyder på att Duration fungerar som en rimlig exponeringsvariabel. För senare modellering betyder det att vi sannolikt bör använda duration som offset eller exponering, snarare än att bara lägga in den som en vanlig förklaringsvariabel.

## 8. Samband mellan storlek och skador

Här delas observationerna in i storleksgrupper för att undersöka hur skadefrekvensen varierar med omsättning, försäkringsbelopp och självrisk.

In [ ]:
turnover_deciles = pd.qcut(df["Omsattning"], q=10, duplicates="drop")
insured_amount_deciles = pd.qcut(df["Forsakringsbelopp"], q=10, duplicates="drop")
deductible_groups = pd.qcut(df["Sjalvrisk"], q=10, duplicates="drop")

turnover_size_summary = (
    df.assign(Omsattningsdecil=turnover_deciles)
    .groupby("Omsattningsdecil", observed=False)
    .agg(
        **{
            "Antal rader": ("Omsattning", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
)

insured_amount_size_summary = (
    df.assign(Forsakringsbeloppsdecil=insured_amount_deciles)
    .groupby("Forsakringsbeloppsdecil", observed=False)
    .agg(
        **{
            "Antal rader": ("Forsakringsbelopp", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
)

deductible_size_summary = (
    df.assign(Sjalvrisksgrupp=deductible_groups)
    .groupby("Sjalvrisksgrupp", observed=False)
    .agg(
        **{
            "Antal rader": ("Sjalvrisk", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
)

turnover_size_summary[["Omsattningsdecil", "Skador per exponerat år"]]


### Tolkning

När observationerna delas in i storleksgrupper syns ett tydligt mönster:

- högre omsättning hänger ihop med högre skadefrekvens
- högre försäkringsbelopp hänger också ihop med högre skadefrekvens
- självrisk verkar ha svagare och mindre tydligt samband

Exempelvis stiger skadefrekvensen tydligt mellan de lägsta och högsta decilerna för både omsättning och försäkringsbelopp.

Större företag och större försäkrade värden verkar ha högre risk. Det är affärsmässigt rimligt, men betyder också att modelleringen måste hantera att flera storleksmått bär liknande information.

## 9. Korrelation mellan ekonomiska variabler

Här studeras den linjära samvariationen mellan de ekonomiska variablerna, både i ursprunglig skala och efter log-transformering.

In [ ]:
economic_variables = df[["Omsattning", "Forsakringsbelopp", "Sjalvrisk"]]
pearson_corr = economic_variables.corr(method="pearson")

log_economic_variables = pd.DataFrame(
    {
        "log_Omsattning": np.log1p(df["Omsattning"]),
        "log_Forsakringsbelopp": np.log1p(df["Forsakringsbelopp"]),
        "log_Sjalvrisk": np.log1p(df["Sjalvrisk"]),
    }
)
log_pearson_corr = log_economic_variables.corr(method="pearson")

pearson_corr


### Tolkning

Pearson-korrelation i träningsdatan:

- Omsättning vs Försäkringsbelopp: 0,435
- Omsättning vs Självrisk: 0,456
- Försäkringsbelopp vs Självrisk: 0,213

Med log-transformerade variabler blir sambanden ännu tydligare, särskilt mellan omsättning och försäkringsbelopp.

Detta bekräftar uppgiftens varning om korrelation mellan dessa variabler. Deskriptivt betyder det att stora företag ofta också har höga försäkringsbelopp och högre självrisk. I en senare regressionsmodell kan detta ge:

- svårare tolkning av enskilda koefficienter
- instabilare skattningar
- behov av att jämföra modeller med olika variabeluppsättningar

## 10. De tydligaste riskkombinationerna

Här kombineras verksamhet och geografi för att identifiera de grupper som har högst skadefrekvens bland segment med tillräckligt många observationer.

In [ ]:
min_rows_for_combo = 20000

business_geography_combo = (
    df.groupby(["Verksamhet", "GeografisktOmrade"], observed=False)
    .agg(
        **{
            "Antal rader": ("AntalSkador", "size"),
            "Totala skador": ("AntalSkador", "sum"),
            "Exponeringsår": ("Duration", "sum"),
        }
    )
    .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
    .reset_index()
)

top_risk_combinations = (
    business_geography_combo[business_geography_combo["Antal rader"] >= min_rows_for_combo]
    .sort_values("Skador per exponerat år", ascending=False)
    .reset_index(drop=True)
)

top_risk_combinations[["Verksamhet", "GeografisktOmrade", "Antal rader", "Skador per exponerat år"]].head(10)


### Tolkning

Bland kombinationer med tillräckligt många observationer är de högsta skadefrekvenserna:

- VVS i storstad: 0,0398
- VVS i mellanstorstad: 0,0311
- Takarbeten i storstad: 0,0309
- Byggföretag i storstad: 0,0268
- Övriga specialistföretag i storstad: 0,0266

Det verkar alltså inte bara vara enskilda effekter från verksamhet och geografi, utan möjligen också interaktioner mellan dem. Det är något vi kan nämna redan i analysdelen och eventuellt testa senare i modellen.

## 11. Viktigaste slutsatserna från den deskriptiva analysen

Det här är de mest användbara slutsatserna att bära med sig vidare:

- Skador är sällsynta, vilket gör att modellen bör behandla detta som frekvensdata med många nollor.
- Verksamhet spelar stor roll, särskilt VVS och takarbeten.
- Geografi spelar också stor roll, där storstad har klart högre skadefrekvens.
- Tidseffekter finns, även om de inte är jättestora.
- Duration bör användas som exponering, inte ignoreras.
- Omsättning och försäkringsbelopp verkar kopplas till högre risk, men de är också korrelerade med varandra.
- Portföljen är heterogen, så en enda genomsnittlig skadefrekvens för alla kunder skulle missa viktiga riskskillnader.

## 12. Formulering ni kan använda i rapporten

Ni kan skriva något i stil med:

> Den deskriptiva analysen visar att skadefrekvensen i portföljen är låg totalt sett, men att det finns tydliga skillnader mellan verksamhetstyper, geografiska områden och år. VVS, takarbeten och storstadsmarknader uppvisar högre skadefrekvens än övriga grupper. Samtidigt är ekonomiska variabler som omsättning och försäkringsbelopp tydligt högerskevda och delvis korrelerade. Detta motiverar en modellansats där skadeantal modelleras som count-data med hänsyn till exponering via duration samt med riskklassificering efter verksamhet, geografi och tid.